# Circuit 3 — Quantum Fourier Transform (3 qubits)

**What it does:** Implements the standard 3-qubit QFT using Hadamard gates,
controlled-phase rotations, and a bit-reversal SWAP. The controlled-phase gate
is decomposed into `P(θ/2) · CNOT · P(-θ/2) · CNOT · P(θ/2)` — no new IR
gates needed.

**Two-circuit approach:** `ManyShotRunner` requires Clifford gates (Stim backend).
P(θ) for arbitrary θ is non-Clifford, so the heatmap/GIF pass uses a Clifford
stand-in. Where the angle is exactly π/2 we use the native CS gate (exact Clifford);
for non-Clifford angles (π/4, π/8, …) every P gate is replaced by S or S†
(same gate count and CNOT topology). The purity pass uses the real QFT circuit
via `TrajectoryBackend`.

**Phase angles used:**
- Between q0–q1: θ = π/2 → **CS gate** (exact Clifford)
- Between q0–q2: θ = π/4 → T gate — non-Clifford → S/S†/CNOT stand-in
- Between q1–q2: θ = π/2 → **CS gate** (exact Clifford)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

N_SHOTS      = 2000
N_SHOTS_TRAJ = 400

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
# ── Gate helpers ──────────────────────────────────────────────────────────────

def controlled_phase(c: nq.Circuit, ctrl: int, tgt: int,
                     theta: float, t=None) -> None:
    """Decompose ctrl-P(θ) into 4 standard gates."""
    c.p(ctrl, theta / 2, t=t)
    c.cnot(ctrl, tgt)
    c.p(tgt, -theta / 2)
    c.cnot(ctrl, tgt)
    c.p(tgt, theta / 2)


def controlled_phase_clifford(c: nq.Circuit, ctrl: int, tgt: int,
                               t=None) -> None:
    """Clifford stand-in: replaces P(θ) with S / S†, keeps CNOT structure."""
    c.s(ctrl, t=t)
    c.cnot(ctrl, tgt)
    c.s_dag(tgt)
    c.cnot(ctrl, tgt)
    c.s(tgt)


# ── Shared pass helpers ───────────────────────────────────────────────────────

def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    result_traj = TrajectoryBackend().run(
        circuit, noise_model=noise_config_twirl, n_shots=N_SHOTS_TRAJ, seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities,
                      label, gif_name):
    fig = plot_error_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.many_shot_result = result_many
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_QFT = 3

def build_qft_clifford(n: int) -> nq.Circuit:
    """
    Clifford stand-in for ManyShotRunner heatmap/GIF.
    - ctrl-P(π/2) (j−k=1): exact Clifford → native CS gate.
    - ctrl-P(θ<π/2) (j−k>1): non-Clifford angle → S/S†/CNOT stand-in.
    """
    c = nq.Circuit(n_qubits=n, name=f"qft_clifford_{n}q")
    for k in range(n):
        c.h(k)
        for j in range(k + 1, n):
            if j - k == 1:
                c.cs(k, j)  # ctrl-P(π/2) = CS — exact Clifford gate
            else:
                controlled_phase_clifford(c, ctrl=k, tgt=j)  # non-Clifford stand-in
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c


def build_qft_circuit(n: int) -> nq.Circuit:
    """
    Real n-qubit QFT for TrajectoryBackend purity pass.
    Structure: for each qubit k, apply H then controlled-P(π/2^(j−k)) toward
    each later qubit j. Finish with bit-reversal SWAPs.
    """
    c = nq.Circuit(n_qubits=n, name=f"qft_{n}q")
    for k in range(n):
        c.h(k)
        for j in range(k + 1, n):
            theta = np.pi / (2 ** (j - k))
            controlled_phase(c, ctrl=k, tgt=j, theta=theta)
    for i in range(n // 2):
        c.swap(i, n - 1 - i)
    return c


circuit_qft_clifford = fill_idle_with_identities(
    build_qft_clifford(N_QFT), gate_times
)
circuit_qft = fill_idle_with_identities(
    build_qft_circuit(N_QFT), gate_times
)

print(f"Clifford circuit ops: {len(circuit_qft_clifford.operations)}")
print(f"Real QFT circuit ops: {len(circuit_qft.operations)}")

In [ ]:
noise_qft_clifford = profile.to_noise_model(
    circuit_qft_clifford, mode="t2", representation="pauli_twirl",
)
noise_qft = profile.to_noise_model(
    circuit_qft, mode="t2", representation="pauli_twirl",
)

result_qft = ManyShotRunner().run(
    circuit_qft_clifford, n_shots=N_SHOTS, noise_config=noise_qft_clifford, seed=42,
)
print(f"ManyShotRunner done  zero-error fraction: {result_qft.zero_error_fraction:.4f}")

rho_qft, pur_qft = run_purity_pass(circuit_qft, noise_qft, N_QFT, "QFT-3")

In [ ]:
# Heatmap and GIF use the Clifford stand-in circuit.
# Purity panel uses rho from the real QFT TrajectoryBackend pass.
visualize_circuit(
    circuit_qft_clifford, result_qft, noise_qft_clifford, rho_qft, pur_qft,
    label="QFT (3q)",
    gif_name="qft_3q",
)